In [ ]:
from mask import *
import cv2
import numpy as np
import time
import uiautomator2 as u2
d = u2.connect('emulator-5554')
def swipe_car():
    d.touch.down(186, 670) # 模拟按下
    time.sleep(.01) # down 和 move 之间的延迟，自己控制
    for i in range(39):
        d.touch.move(186-i*2, 670) #
    time.sleep(0.5) # down 和 move 之间的延迟，自己控制
    d.touch.up(186-i*2, 670) # 模拟抬起
    time.sleep(1)
    d.click(70,670)


In [ ]:
import uiautomator2 as u2
d = u2.connect('adb-fc65396d-4LPqmI._adb-tls-connect._tcp')
#!/usr/bin/env python3
# pick_hsv_gui.py
# 需求: python3, opencv-python, numpy (tkinter 通常已內建)
#
# 用法:
#   python pick_hsv_gui.py
#
# 操作說明 (執行後):
# - 會跳出檔案選擇視窗，選一張圖片
# - 在 "Original" 視窗中點擊取得該像素的 HSV 與 BGR（會在視窗上顯示）
# - 用 H_min / H_max / S_min / S_max / V_min / V_max 調整遮罩
# - 按 'c' 複製目前顯示的 HSV 值（同時會把值寫入 pick_hsv_result.json）
# - 按 'q' 或 ESC 關閉程式

import cv2
import numpy as np
import json
import os
from tkinter import Tk, filedialog

# ------- helper / UI callbacks -------
state = {
    "img_bgr": None,
    "img_hsv": None,
    "sample_hsv": None,
    "sample_bgr": None,
    "window_name_orig": "Original",
    "window_name_mask": "Mask",
    "window_name_result": "Result",
}

def nothing(x):
    pass

def open_image_dialog():
    Tk().withdraw()
    path = filedialog.askopenfilename(
        title="Select image",
        filetypes=[("Image files", "*.png;*.jpg;*.jpeg;*.bmp;*.tiff;*.webp"), ("All files","*.*")]
    )
    return path

def on_mouse(event, x, y, flags, param):
    if state["img_bgr"] is None:
        return
    if event == cv2.EVENT_LBUTTONDOWN:
        bgr = state["img_bgr"][y, x].tolist()  # BGR order
        hsv = cv2.cvtColor(np.uint8([[bgr]]), cv2.COLOR_BGR2HSV)[0,0].tolist()
        state["sample_bgr"] = tuple(int(v) for v in bgr)
        state["sample_hsv"] = tuple(int(v) for v in hsv)
        # print to console as well
        print(f"Clicked at ({x},{y})  BGR={state['sample_bgr']}  HSV={state['sample_hsv']}")

def draw_overlay(img):
    """Draw sample HSV/BGR text overlay on image copy"""
    out = img.copy()
    if state["sample_hsv"] is not None:
        h,s,v = state["sample_hsv"]
        b,g,r = state["sample_bgr"]
        txt1 = f"HSV: ({h}, {s}, {v})"
        txt2 = f"BGR: ({b}, {g}, {r})"
        # draw background rectangle
        cv2.rectangle(out, (5,5), (380,60), (0,0,0), -1)
        cv2.putText(out, txt1, (10,25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2, cv2.LINE_AA)
        cv2.putText(out, txt2, (10,50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1, cv2.LINE_AA)
    return out

def load_image(path):
    img = d.screenrshot(format='opencv')
    # cv2.imread on Windows may fail with unicode path; using imdecode+fromfile handles that.
    if img is None:
        raise RuntimeError("無法讀取圖片，請確認檔案路徑與格式。")
    return img

# ------- main -------
def main():
    print("Pick HSV GUI - starting...")
    img_path = open_image_dialog()
    if not img_path:
        print("No file selected. Exiting.")
        return

    img_bgr = load_image(img_path)
    state["img_bgr"] = img_bgr
    state["img_hsv"] = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # create windows
    cv2.namedWindow(state["window_name_orig"], cv2.WINDOW_NORMAL)
    cv2.namedWindow(state["window_name_mask"], cv2.WINDOW_NORMAL)
    cv2.namedWindow(state["window_name_result"], cv2.WINDOW_NORMAL)
    cv2.setMouseCallback(state["window_name_orig"], on_mouse)

    # create trackbars
    cv2.createTrackbar("H_min", state["window_name_mask"], 0, 179, nothing)
    cv2.createTrackbar("H_max", state["window_name_mask"], 179, 179, nothing)
    cv2.createTrackbar("S_min", state["window_name_mask"], 0, 255, nothing)
    cv2.createTrackbar("S_max", state["window_name_mask"], 255, 255, nothing)
    cv2.createTrackbar("V_min", state["window_name_mask"], 0, 255, nothing)
    cv2.createTrackbar("V_max", state["window_name_mask"], 255, 255, nothing)

    # pre-set if sample exists? none yet

    while True:
        # read trackbars
        h_min = cv2.getTrackbarPos("H_min", state["window_name_mask"])
        h_max = cv2.getTrackbarPos("H_max", state["window_name_mask"])
        s_min = cv2.getTrackbarPos("S_min", state["window_name_mask"])
        s_max = cv2.getTrackbarPos("S_max", state["window_name_mask"])
        v_min = cv2.getTrackbarPos("V_min", state["window_name_mask"])
        v_max = cv2.getTrackbarPos("V_max", state["window_name_mask"])

        lower = np.array([h_min, s_min, v_min], dtype=np.uint8)
        upper = np.array([h_max, s_max, v_max], dtype=np.uint8)

        # mask (注意 H 的環繞情況：若 h_min > h_max 代表範圍跨越 0)
        if h_min <= h_max:
            mask = cv2.inRange(state["img_hsv"], lower, upper)
        else:
            # wrap-around: H in [h_min,179] U [0,h_max]
            lower1 = np.array([h_min, s_min, v_min], dtype=np.uint8)
            upper1 = np.array([179, s_max, v_max], dtype=np.uint8)
            lower2 = np.array([0, s_min, v_min], dtype=np.uint8)
            upper2 = np.array([h_max, s_max, v_max], dtype=np.uint8)
            mask1 = cv2.inRange(state["img_hsv"], lower1, upper1)
            mask2 = cv2.inRange(state["img_hsv"], lower2, upper2)
            mask = cv2.bitwise_or(mask1, mask2)

        # result: apply mask to original
        result = cv2.bitwise_and(state["img_bgr"], state["img_bgr"], mask=mask)

        # overlay sample info on original
        orig_display = draw_overlay(state["img_bgr"])
        # scale windows to fit screen? using WINDOW_NORMAL allows resizing by user

        cv2.imshow(state["window_name_orig"], orig_display)
        cv2.imshow(state["window_name_mask"], mask)
        cv2.imshow(state["window_name_result"], result)

        key = cv2.waitKey(30) & 0xFF
        if key == ord('q') or key == 27:  # q or ESC
            break
        elif key == ord('c'):
            # copy current sample hsv and the current trackbar range to a json
            payload = {
                "clicked_hsv": state["sample_hsv"],
                "clicked_bgr": state["sample_bgr"],
                "h_range": [h_min, h_max],
                "s_range": [s_min, s_max],
                "v_range": [v_min, v_max],
                "image_path": os.path.abspath(img_path)
            }
            out_path = os.path.join(os.getcwd(), "pick_hsv_result.json")
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(payload, f, ensure_ascii=False, indent=2)
            print(f"Saved HSV data to {out_path}")
        elif key == ord('p'):
            # print currently selected HSV (if any)
            print("Current sample HSV:", state.get("sample_hsv"))
            print("Current trackbar ranges:", {"H":[h_min,h_max],"S":[s_min,s_max],"V":[v_min,v_max]})

    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()



Pick HSV GUI - starting...


In [ ]:
import cv2
import requests
import base64
OCR_SERVER_URL = "http://100.64.0.5:5000"  # OCR 服務器地址
def encode_image(img):
    """將圖片編碼為 base64"""
    _, buffer = cv2.imencode('.jpg', img)
    img_base64 = base64.b64encode(buffer).decode('utf-8')
    return img_base64

def analyze_skill_via_http(img_roi):
    """透過 HTTP 分析技能"""
    try:
        img_base64 = encode_image(img_roi)
        response = requests.post(
            f"{OCR_SERVER_URL}/analyze_skill",
            json={'image': img_base64},
            timeout=10
        )
        
        if response.status_code == 200:
            return response.json()
        else:
            print(f"HTTP 錯誤: {response.status_code}")
            return None
    except Exception as e:
        print(f"HTTP 請求失敗: {e}")
        return None
img  = cv2.imread(r"a.jpg")

result = analyze_skill_via_http(img)
print(result)

{'combo': '', 'is_unwanted': False, 'normalized_combo': '', 'ocr_results': ['穿越深淵之門'], 'success': True}


In [ ]:

full_park()

True

In [137]:
img = d.screenshot(format='opencv')
result = reader.readtext(img)
for i in result:
    print(i)
    print("----")

([[255, 3], [279, 3], [279, 17], [255, 17]], '0.0', 0.9583385068651308)
----
([[385, 3], [409, 3], [409, 19], [385, 19]], 'PrS', 0.5042912321260506)
----
([[463, 3], [511, 3], [511, 19], [463, 19]], 'Size: 0.', 0.6203983294339525)
----
([[464, 58], [534, 58], [534, 82], [464, 82]], '05.26:35', 0.8604534634253949)
----
([[185, 54], [354, 54], [354, 110], [185, 110]], '停車管理', 0.9076226410325715)
----
([[50, 136], [146, 136], [146, 162], [50, 162]], '自助小摩托', 0.6909075641124874)
----
([[48, 396], [162, 396], [162, 420], [48, 420]], '奇遇獎勵預覽(', 0.18266193147042878)
----
([[189, 399], [245, 399], [245, 419], [189, 419]], '+45%)', 0.3982041552409864)
----
([[228, 534], [356, 534], [356, 558], [228, 558]], '20/min +110%', 0.47822859582996413)
----
([[228, 566], [346, 566], [346, 590], [228, 590]], '10/min +49%', 0.8952325539800962)
----
([[180, 596], [360, 596], [360, 620], [180, 620]], '今日累計停車時間0分鐘', 0.9139457481087363)
----
([[48, 656], [92, 656], [92, 682], [48, 682]], '它停+', 0.0383972247539

In [ ]:
def park():
    d.click(321,920)
    time.sleep(3)
    d.click(430,401)
    time.sleep(3)
    world_car = check_world_car(d.screenshot(format='opencv'))
    car_num = count_car(d.screenshot(format='opencv'))
    if not world_car:
        pass
        # d.click(267,915)
        # time.sleep(3)
        # d.click(331,796)
        # time.sleep(3)
    if car_num < 5:
        d.click(267,915)
        time.sleep(2)



    print(world_car,car_num)
park()

False 0


In [19]:
img = d.screenshot(format='opencv')
# cv2.imshow("img", img[250:298,50:112])
# cv2.waitKey(0)
# cv2.imwrite("world.jpg",img[250:298,50:112])
cv2.imshow("img", img[:,45:145])
cv2.waitKey(0)

-1

In [2]:
import mask
import cv2
import numpy as np
import uiautomator2 as u2
d = u2.connect('emulator-5554')
def Martial_Soul():
    img = d.screenshot(format='opencv')[472:538,455:528]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower = mask.red_mask_lower
    upper = mask.red_mask_upper
    mask1 = cv2.inRange(hsv, lower, upper)
    # 計算面積
    contours, _ = cv2.findContours(mask1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    num_boxes = 0
    for contour in contours:
        area = cv2.contourArea(contour)
        if area > 50:
            num_boxes += 1
    if num_boxes > 0:
        return True
    return False
def get_Martial_Soul():
    if  Martial_Soul():
        d.click(495,512)
        time.sleep(3)
        d.click(451,792)
        time.sleep(3)
        d.click(289,272)
        time.sleep(3)
        d.click(474,254)
        time.sleep(3)
        d.click(55,48)
        time.sleep(3)
        d.click(486,908)
        time.sleep(3)
        d.click(271,883)

In [ ]:
def check_skill_and_parner():
    img = d.screenshot(format='opencv')
    hsv = cv2.cvtColor(img[876:910,461:541], cv2.COLOR_BGR2HSV)
    lower = mask.red_mask_lower
    upper = mask.red_mask_upper
    mask1 = cv2.inRange(hsv, lower, upper)
    # 計算面積
    contours, _ = cv2.findContours(mask1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    num_boxes = 0
    for contour in contours:
        area = cv2.contourArea(contour)
        if area >100:
            return True
    return False
def skill_and_partner():
    d.click(500,900)
    time.sleep(5)
    d.click(150,547)
    time.sleep(1)
    click_white()
    time.sleep(1)
    d.click(72,830)
    for i in range(5):
        d.click(105,620)
        time.sleep(3)
        d.click(264,890)
        time.sleep(3)
    d.click(352,101)
    time.sleep(1)
    for i in range(5):
        d.click(105,620)
        time.sleep(3)
        d.click(264,890)
        time.sleep(3)
    d.click(500,900)
check_skill_and_parner()

True

In [ ]:
import time
d.click(72, 830)
for i in range(5):
    d.click(105, 620)
    time.sleep(3)
    d.click(264, 890)
    time.sleep(3)
d.click(352, 101)
time.sleep(1)
for i in range(5):
    d.click(105, 620)
    time.sleep(3)
    d.click(264, 890)
    time.sleep(3)
d.click(500, 900)

In [ ]:

check_world_car(cv2.imread(r"A:\Screenshots\Screenshot_20241115-033039.png"))
# img = cv2.imread(r"A:\Screenshots\Screenshot_20241115-033039.png")
# cv2.imshow("img", img[138:259])
# print(img[215,60])
# img=cv2.circle(img, (215,60), 50, (174 , 221  ,243), -1)

# # 使用hsv
# hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
# lower = np.array([9, 90, 139])
# upper = np.array([13, 157, 175])

# mask = cv2.inRange(hsv, lower, upper)
# cv2.imshow("mask1", mask[138:259])
# # 膨脹後腐蝕
# mask = cv2.dilate(mask, None, iterations=2)

# mask = cv2.erode(mask, None, iterations=2)

# contours, _ = cv2.findContours(mask[138:259], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# # 計算框框的數量
# num_boxes = 0
# for contour in contours:
#     # 計算輪廓的面積過濾小的噪點
#     area = cv2.contourArea(contour)
#     if area > 100:  # 調整此閾值以忽略小的雜訊
#         num_boxes += 1

# print(f"框框的數量：{num_boxes}")

# # 顯示偵測框框的結果
# output = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
# cv2.drawContours(output, contours, -1, (0, 255, 0), 2)
# cv2.imshow("Detected Boxes", output)
# cv2.waitKey(0)
# cv2.destroyAllWindows()




# cv2.imshow("mask", mask[138:259])
# cv2.waitKey(0)
# 使用cv2.findContours找到輪廓


# # cv2.imwrite("full.jpg",img[300:320,42:79])
# find = cv2.imread("full.jpg")
# res = cv2.matchTemplate(img, find, cv2.TM_CCOEFF_NORMED)
# loc = np.where(res >= 0.8)
# # 繪製矩形
# for pt in zip(*loc[::-1]):
#     cv2.rectangle(img, pt, (pt[0] + find.shape[1], pt[1] + find.shape[0]), (0, 0, 255), 2)
# cv2.imshow("img", img)
# cv2.waitKey(0)
# if len(loc[0]) > 0:
#     center = [int(loc[1][0] + find.shape[1] / 2), int(loc[0][0] + find.shape[0] / 2)]
#     print(center)

    # d.click(center[0], center[1])
    # time.sleep(5)
    # return True


1

In [6]:
import uiautomator2 as u2
import cv2
import numpy as np

# 連接到設備
d = u2.connect('emulator-5568')

# 捕捉屏幕截圖
img = d.screenshot(format='opencv')
# conditions = [abs(np.sum(img[955, 535]) - np.sum([47, 138, 123])) <= 10,abs(np.sum(img[902, 39]) - np.sum([146, 232, 232])) <= 10,abs(np.sum(img[956, 6]) - np.sum([50, 140, 117])) <= 10,abs(np.sum(img[921, 135]) - np.sum([41, 21, 218])) <= 10,abs(np.sum(img[908, 223]) - np.sum([160, 165, 164])) <= 10,abs(np.sum(img[731, 27]) - np.sum([139, 170, 201])) <= 10,abs(np.sum(img[759, 30]) - np.sum([111, 143, 179])) <= 10,abs(np.sum(img[794, 37]) - np.sum([38, 60, 88])) <= 10,abs(np.sum(img[825, 380]) - np.sum([37, 58, 86])) <= 10]
# print(all(conditions))
# 定義滑鼠點擊事件的回調函數
def on_mouse_click(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        # 獲取點擊位置的顏色
        color = img[y, x]
        print(f"點擊位置 ({x}, {y}) 的顏色: {color}")

# 顯示圖片並設置滑鼠回調函數
cv2.imshow('Screenshot', img)
cv2.setMouseCallback('Screenshot', on_mouse_click)

# 等待按鍵事件
cv2.waitKey(0)
cv2.destroyAllWindows()



In [9]:
img = d.screenshot(format='opencv')[638:700,263:327]
cv2.imshow("img", img)
cv2.waitKey(0)
cv2.imwrite("unpark1.jpg",img)

True

In [ ]:

conditions = [abs(np.sum(img[955, 535]) - np.sum([47, 138, 123])) <= 10,abs(np.sum(img[902, 39]) - np.sum([146, 232, 232])) <= 10,abs(np.sum(img[956, 6]) - np.sum([50, 140, 117])) <= 10,abs(np.sum(img[921, 135]) - np.sum([41, 21, 218])) <= 10,abs(np.sum(img[908, 223]) - np.sum([160, 165, 164])) <= 10,abs(np.sum(img[731, 27]) - np.sum([139, 170, 201])) <= 10,abs(np.sum(img[759, 30]) - np.sum([111, 143, 179])) <= 10,abs(np.sum(img[794, 37]) - np.sum([38, 60, 88])) <= 10,abs(np.sum(img[825, 380]) - np.sum([37, 58, 86])) <= 10]

In [ ]:
conditions= [abs(np.sum(img[576,375]) - np.sum([180, 208, 219])) <= 10,abs(np.sum(img[700,121]) - np.sum([178, 209, 218])) <= 10,abs(np.sum(img[795,408]) - np.sum([42, 155, 111])) <= 10,abs(np.sum(img[790,217]) - np.sum([58, 65, 198])) <= 10]
點擊位置 (375, 576) 的顏色: [180 208 219]
點擊位置 (121, 700) 的顏色: [178 209 218]
點擊位置 (408, 795) 的顏色: [ 42 155 111]
點擊位置 (217, 790) 的顏色: [ 58  65 198]


點擊位置 (221, 554) 的顏色: [ 58  65 198]
點擊位置 (424, 550) 的顏色: [ 42 154 112]
conditions = [abs(np.sum(img[554, 221]) - np.sum([58, 65, 198])) <= 10,abs(np.sum(img[550, 424]) - np.sum([42, 154, 112])) <= 10]
點擊位置 (77, 267) 的顏色: [195 225 236]
點擊位置 (381, 267) 的顏色: [206 237 246]
點擊位置 (476, 860) 的顏色: [244 255 255]
conditions = [abs(np.sum(img[267, 77]) - np.sum([195, 225, 236])) <= 10,abs(np.sum(img[267, 381]) - np.sum([206, 237, 246])) <= 10,abs(np.sum(img[860, 476]) - np.sum([244, 255, 255])) <= 10]

False


In [ ]:
import random
img = d.screenshot(format='opencv')
conditions =[abs(np.sum(img[231,126]) - np.sum([137, 207, 220])) < 10, abs(np.sum(img[245,258]) - np.sum([76, 99, 191])) < 10, abs(np.sum(img[276,385]) - np.sum([23, 41, 112])) < 10, abs(np.sum(img[316,469]) - np.sum([47, 102, 163])) < 10, abs(np.sum(img[229,422]) - np.sum([95, 122, 203])) < 10, abs(np.sum(img[321,168]) - np.sum([215, 234, 255])) < 10, abs(np.sum(img[285,140]) - np.sum([192, 241, 255])) < 10, abs(np.sum(img[297,269]) - np.sum([108, 152, 241])) < 10, abs(np.sum(img[312,269]) - np.sum([109, 165, 236])) < 10, abs(np.sum(img[346,289]) - np.sum([175, 213, 225])) < 10, abs(np.sum(img[327,304]) - np.sum([111, 176, 221])) < 10, abs(np.sum(img[213,474]) - np.sum([3, 24, 216])) < 10]

if all(conditions):
    d.click(random.randint(78,452), random.randint(218,330))
    time.sleep(2)
    d.click(357,900)
    time.sleep(2)
    for i in range(7):
        d.click(273,300)
        time.sleep(1)


In [ ]:
import easyocr
reader = easyocr.Reader(['ch_tra', 'en'])


In [29]:
d = u2.connect('emulator-5562')
# img = d.screenshot(format='opencv')
def Martial_Soul(d):
    img = d.screenshot(format='opencv')[472:538,455:528]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower = mask.red_mask_lower
    upper = mask.red_mask_upper
    mask1 = cv2.inRange(hsv, lower, upper)
    # 計算面積
    contours, _ = cv2.findContours(mask1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    num_boxes = 0
    for contour in contours:
        area = cv2.contourArea(contour)
        if area > 50:
            num_boxes += 1
    if num_boxes > 0:
        return True
    return False
def Martial_Soul2(d):
    img = d.screenshot(format='opencv')[760:791,483:525]
    cv2.imshow("img", img)
    cv2.waitKey(0)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower = mask.red_mask_lower
    upper = mask.red_mask_upper
    mask1 = cv2.inRange(hsv, lower, upper)
    # 計算面積
    contours, _ = cv2.findContours(mask1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    num_boxes = 0
    for contour in contours:
        area = cv2.contourArea(contour)
        if area > 50:
            num_boxes += 1
    if num_boxes > 0:
        return True
    return False
Martial_Soul2(d)


False

In [ ]:
def park():
    d.click(321,920)
    time.sleep(3)
    d.click(430,401)
    check_world_car(d.screenshot(format='opencv'))
park()

In [ ]:
import uiautomator2 as u2
import cv2
import numpy as np
import random
import time
from mask import *
d= u2.connect('emulator-5554')
# 讀取圖像
image = d.screenshot(format='opencv')

def click_white():
    d.click(509,56)
    time.sleep(1)
def open_red_envelope(d):
    d.click(random.randint(50, 100), random.randint(843, 848))
    time.sleep(2)
    d.click(random.randint(25, 79), random.randint(684, 753))
    time.sleep(2)
    for i in range(3):
        d.click(random.randint(41, 135), random.randint(237, 312))
        time.sleep(random.random())
        d.click(random.randint(46, 76), random.randint(783, 818))
        time.sleep(random.random())
    click_white(d)


def check_red_in_pic(img):
    hsv_image = cv2.cvtColor(img[814:862,0:56], cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv_image,red_mask_lower, red_mask_upper)
    if np.sum(mask) > 0:
        return True
    else:
        return False

img = d.screenshot(format='opencv')[814:862,0:56]
check_red_in_pic(img)


False

: 

In [ ]:
img = d.screenshot(format='opencv')
img = img[279+30:339+30,119+12:180+13]

# img = img[279:339,356:417]

# cv2.imshow("img", img)
# cv2.waitKey(500)
cv2.imwrite("offline_jup.jpg",img)

True

In [ ]:
image = d.screenshot(format='opencv')[881:956, 106:426]
result = cv2.matchTemplate(image, cropped, cv2.TM_CCOEFF_NORMED)
print(cv2.minMaxLoc(result)[1])

{'currentPackageName': 'com.mxdzz.tw.and',
 'displayHeight': 960,
 'displayRotation': 0,
 'displaySizeDpX': 360,
 'displaySizeDpY': 640,
 'displayWidth': 540,
 'productName': 'SM-N976N',
 'screenOn': True,
 'sdkInt': 28,
 'naturalOrientation': True}

In [2]:
import cv2
import numpy as np
import uiautomator2 as u2

d= u2.connect('emulator-5554')
# 讀取圖像
image = d.screenshot(format='opencv')
# image = cv2.imread(r'A:\Screenshots\Screenshot_20241111-210050.png')
hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

# 回調函數，trackbar 需要
def nothing(x):
    pass

# 建立窗口
cv2.namedWindow("Mask Adjuster")

# 創建 Trackbars 用於調整 HSV 範圍
cv2.createTrackbar("Low H", "Mask Adjuster", 0, 179, nothing)
cv2.createTrackbar("High H", "Mask Adjuster", 179, 179, nothing)
cv2.createTrackbar("Low S", "Mask Adjuster", 0, 255, nothing)
cv2.createTrackbar("High S", "Mask Adjuster", 255, 255, nothing)
cv2.createTrackbar("Low V", "Mask Adjuster", 0, 255, nothing)
cv2.createTrackbar("High V", "Mask Adjuster", 255, 255, nothing)

while True:
    # 獲取 Trackbars 的當前值
    low_h = cv2.getTrackbarPos("Low H", "Mask Adjuster")
    high_h = cv2.getTrackbarPos("High H", "Mask Adjuster")
    low_s = cv2.getTrackbarPos("Low S", "Mask Adjuster")
    high_s = cv2.getTrackbarPos("High S", "Mask Adjuster")
    low_v = cv2.getTrackbarPos("Low V", "Mask Adjuster")
    high_v = cv2.getTrackbarPos("High V", "Mask Adjuster")

    # 定義 HSV 顏色範圍
    lower_bound = np.array([low_h, low_s, low_v])
    upper_bound = np.array([high_h, high_s, high_v])

    # 產生遮罩
    mask = cv2.inRange(hsv_image, lower_bound, upper_bound)
    masked_image = cv2.bitwise_and(image, image, mask=mask)

    # 顯示遮罩與結果
    cv2.imshow("Mask", mask)
    cv2.imshow("Masked Image", masked_image)

    # 按下 'q' 鍵退出
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 釋放所有資源
cv2.destroyAllWindows()


SyntaxError: invalid syntax (4061933866.py, line 1)

In [ ]:
d.shell("am start -a android.intent.action.VIEW -d https://www.google.com")

KeyboardInterrupt: 

In [11]:
img = d.screenshot(format='opencv')

result = easyocr_reader.readtext(img)
for i in result:
    print(i)
    print("----")

NameError: name 'easyocr_reader' is not defined

In [ ]:
!pip3 install pytesseract


In [1]:
import pytesseract
from PIL import Image
import uiautomator2 as u2
from paddleocr import PaddleOCR,draw_ocr
from matplotlib import pyplot as plt

import cv2 #opencv
import os

d = u2.connect('emulator-5554')



In [2]:
import easyocr

reader = easyocr.Reader(['ch_tra', 'en'])


In [60]:
img = d.screenshot(format='opencv')

results = reader.readtext(img)
for i in results:
    print(i)
    print("----")

([[1, 3], [49, 3], [49, 19], [1, 19]], 'P: 0/1', 0.5071967141154511)
----
([[77, 3], [139, 3], [139, 19], [77, 19]], 'dX: -16.8', 0.47320121896303663)
----
([[155, 3], [219, 3], [219, 19], [155, 19]], 'd/: 224.0', 0.9758553670893548)
----
([[231, 3], [295, 3], [295, 19], [231, 19]], 'X+: 0.116', 0.2873957588631936)
----
([[309, 1], [371, 1], [371, 19], [309, 19]], '/1: 0.289', 0.8674810237103849)
----
([[463, 3], [509, 3], [509, 19], [463, 19]], 'Size: 0', 0.7976521411924712)
----
([[467, 61], [535, 61], [535, 79], [467, 79]], '04:36!40', 0.8154604107742436)
----
([[200, 82], [344, 82], [344, 106], [200, 106]], '|s14811跨界車位11', 0.3028065622240772)
----
([[76, 100], [150, 100], [150, 124], [76, 124]], '3434.6K', 0.9868750193050219)
----
([[220, 100], [258, 100], [258, 124], [220, 124]], '158', 0.8288207427236006)
----
([[12, 146], [88, 146], [88, 172], [12, 172]], '跨界車位', 0.8968321681022644)
----
([[123, 153], [175, 153], [175, 169], [123, 169]], '100.!00$', 0.24558152463880845)
----
([

In [75]:
# [114, 268], [226, 268], [226, 298], [114, 298]
# img = d.screenshot(format='opencv',filename='t.jpg')[200::,:]
img = d.screenshot(format='opencv')[200::,:]
results = reader.readtext(img)
for i in results:
    print(i)
    print("----")
for result in results:
    if "跨界車位" in result[1]:
        print(result)
        # 找到最小的x,y 和最大的x,y
        x1 = result[0][0][0]
        y1 = result[0][0][1]
        x2 = result[0][2][0]
        y2 = result[0][2][1]

        # [114, 268], [226, 268], [226, 298], [114, 298]
        cv2.rectangle(img, (x1, y1), (x2+60, y2+30), (0, 0, 255), 2)
        cv2.imshow("img", img)
        cv2.waitKey(0)
        # 裁減圖片
        img2 =  img[y1:y2+30,x1:x2+60]
        results = reader.readtext(img2)
        for result in results:
            print(result)
    # if "免戰中" in result[1]:
    #     print(result)
    #     # 找到最小的x,y 和最大的x,y
    #     x1 = result[0][0][0]
    #     y1 = result[0][0][1]
    #     x2 = result[0][2][0]
    #     y2 = result[0][2][1]
    #     print(x1,y1-20,x2,y2)
    #     # [114, 268], [226, 268], [226, 298], [114, 298]
    #     cv2.rectangle(img, (x1, y1-30), (x2, y2), (0, 0, 255), 2)
    #     # 裁減圖片
    #     img2 = img[y1-30:y2,x1:x2]
    #     results = reader.readtext(img2)
    #     for result in results:
    #         print(result)
    else:continue
    print()



([[188, 0], [352, 0], [352, 26], [188, 26]], '[s1474]跨界車位', 0.29458546643831873)
----
([[112, 43], [208, 43], [208, 71], [112, 71]], '跨界車位1', 0.8762666734411675)
----
([[321, 47], [371, 47], [371, 65], [321, 65]], '+100%', 0.9999218442255111)
----
([[114, 68], [226, 68], [226, 96], [114, 96]], '免戰中1:35', 0.6109973167398034)
----
([[321, 75], [371, 75], [371, 91], [321, 91]], '+100%', 0.99994050358235)
----
([[188, 96], [240, 96], [240, 126], [188, 126]], 'Xo', 0.42549501386546756)
----
([[325, 103], [367, 103], [367, 119], [325, 119]], '+50%', 0.9998923540115356)
----
([[112, 132], [212, 132], [212, 162], [112, 162]], '跨界車位3', 0.5004343263431518)
----
([[321, 135], [371, 135], [371, 153], [321, 153]], '+100%', 0.9999607622660986)
----
([[114, 156], [226, 156], [226, 184], [114, 184]], '免戰中03;19', 0.552960006589198)
----
([[321, 165], [371, 165], [371, 181], [321, 181]], '+100%', 0.9999536184199905)
----
([[188, 186], [238, 186], [238, 212], [188, 212]], '% 0', 0.1112755882121081)
----


In [36]:
d = u2.connect('emulator-5554')

ModuleNotFoundError: No module named 'transformers'

In [1]:
from transformers import pipeline

pipe = pipeline("image-to-text", model="DaMax96/Stick_OCR_v3")

c:\Users\eric\anaconda3\envs\nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
config.json: 100%|██████████| 4.89k/4.89k [00:00<00:00, 2.02MB/s]
c:\Users\eric\anaconda3\envs\nlp\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\eric\.cache\huggingface\hub\models--DaMax96--Stick_OCR_v3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to

In [5]:
!pip install opencv-python


  Using cached opencv_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.10.0.84-cp37-abi3-win_amd64.whl (38.8 MB)


In [47]:
import cv2
#讀取一張圖片 裁減 64*62 原圖為65*65

img = cv2.imread('unpark4.jpg')
img = img[1:65,1:63]
cv2.imwrite('unpark4.jpg',img)



True

In [2]:
import subprocess
import time
def run_adb(cmd: str, device_serial: str = None) -> str:
    """
    在终端执行 adb 指令并返回输出。
    如果指定了 device_serial，就用 -s 参数锁定设备。
    """
    base = ['adb']
    if device_serial:
        base += ['-s', device_serial]
    # 将整个 cmd 字符串拆分为列表
    full_cmd = base + cmd.split()
    # 执行并捕获输出
    result = subprocess.run(full_cmd,
                            stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE,
                            text=True,
                            check=False)
    if result.returncode != 0:
        raise RuntimeError(f"ADB Error: {result.stderr.strip()}")
    return result.stdout.strip()

if __name__ == '__main__':
#     adb shell am start \
#   -a android.intent.action.CHOOSER \
#   --es android.intent.extra.TITLE "请选择要打开的菇勇者传说" \
#   --eu android.intent.extra.INTENT \
#     "intent:#Intent;action=android.intent.action.MAIN;category=android.intent.category.LAUNCHER;package=com.mxdzz.tw.and;end"

    # 举例：获取已连接设备列表
    # print(run_adb('devices'))

    # # 举例：在指定设备上启动某个 Activity
    # run_adb(
    #     'shell wm density 240 && wm size 540x960',
    #     device_serial='fc65396d'
    # )
    # # 還原
    # time.sleep(1*7
    #            )
    run_adb(
        'shell wm density reset && wm size reset',
        device_serial='fc65396d'
    )
    #打開包名為org.autojs.autoxjs.v6 不使用/org.autojs.autojs.v6.MainActivity
    # out = run_adb(
    #     'shell monkey -p org.autojs.autoxjs.v6 -c android.intent.category.LAUNCHER 1',
    #     device_serial='fc65396d'
    # )

In [1]:
import subprocess
import shlex

def run_adb(cmd: str, device_serial: str = None) -> str:
    """
    在终端执行 adb 指令并返回输出。
    如果指定了 device_serial，就用 -s 参数锁定设备。
    """
    base = ['adb']
    if device_serial:
        base += ['-s', device_serial]
    # 使用 shlex.split 正确处理引号、空格
    args = shlex.split(cmd)
    full_cmd = base + args
    # 执行并捕获输出
    result = subprocess.run(
        full_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False
    )
    if result.returncode != 0:
        raise RuntimeError(f"ADB Error: {result.stderr.strip()}")
    return result.stdout.strip()

if __name__ == "__main__":
    # 1) 如果只连一台设备，device_serial 可以留 None
    device_serial = None
    # 2) 将那条 shell 命令写成单行字串
    cmd = (
        'shell am start '
        '-a android.intent.action.CHOOSER '
        '--es android.intent.extra.TITLE "请选择要打开的菇勇者传说" '
        '--eu android.intent.extra.INTENT '
        '"intent:#Intent;'
          'action=android.intent.action.MAIN;'
          'category=android.intent.category.LAUNCHER;'
          'package=com.mxdzz.tw.and;'
        'end"'
    )

    try:
        output = run_adb(cmd, device_serial)
        print("ADB 返回：", output)
    except RuntimeError as e:
        print("执行失败：", e)


FileNotFoundError: [WinError 2] 系統找不到指定的檔案。

In [ ]:
import uiautomator2 as u2
d = u2.connect('fc65396d')
d.unlock()

In [1]:
run_adb(
                'shell wm density reset && wm size reset',
                device_serial='fc65396d')

NameError: name 'run_adb' is not defined

In [ ]:
import time
import uiautomator2 as u2

d = u2.connect('3a8d31f2')
def sea(ip,d):
    if "emulator" not in ip:
        d.click(45,360)
    else:
        d.click(45,320)
    time.sleep(10)
    d.click(50,775)
    time.sleep(2)
    d.click(284,712)
    time.sleep(2)
    d.click(519,28) # click white
    time.sleep(2)
    d.click(267,881)
    time.sleep(2)
    d.click(200,917)
    time.sleep(2)
    d.click(263,785)
    time.sleep(2)
    d.click(519,28) # click white
    d.click(519,28) # click white
    time.sleep(1)
    d.click(500,900) #close
    time.sleep(3)
    d.click(70,913)
    time.sleep(2)
    d.click(297,929)
    time.sleep(3)
    d.click(249,814)
    time.sleep(3)
    d.click(500,900) #close
    time.sleep(1)
    d.click(500,900) #close
ip ='fc65396d'
sea(ip,d)

In [37]:
import time
import uiautomator2 as u2

d = u2.connect('3a8d31f2')

In [46]:
d.info.get('screenOn')

False